<a href="https://colab.research.google.com/github/cosmicoxytocin/Notebooks/blob/main/rebasin_mini_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title mount google drive
from google.colab import drive
import os
import os.path as osp
import sys

print("Mounting Google Drive...")
drive.mount('/content/drive')

REPO_PATH = '/content/sd_mecha'
if not osp.exists(REPO_PATH):
    print("Cloning sd-mecha rebasin branch...")
    !git clone --branch rebasin https://github.com/ljleb/sd-mecha {REPO_PATH}
else:
    print("sd-mecha already exists. Skipping...")

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

Mounting Google Drive...
Mounted at /content/drive
Cloning sd-mecha rebasin branch...
Cloning into '/content/sd_mecha'...
remote: Enumerating objects: 2793, done.
remote: Counting objects: 100% (797/797), done.
remote: Compressing objects: 100% (360/360), done.
remote: Total 2793 (delta 641), reused 475 (delta 435), pack-reused 1996 (from 3)
Receiving objects: 100% (2793/2793), 1.01 MiB | 3.92 MiB/s, done.
Resolving deltas: 100% (1663/1663), done.


In [ ]:
#@title clone repo
REPO_URL = "https://github.com/ljleb/sd-mecha"
BRANCH = "rebasin"
TARGET = "/content/sd_mecha"

if not osp.exists(TARGET):
    !git clone -b {BRANCH} {REPO_URL} {TARGET}
else:
    print("Skipping")

if TARGET not in sys.path:
    sys.path.insert(0, {TARGET})

%cd /content/sd-mecha
%pip install -q -e .
%cd /content

In [ ]:
%cd /content/sd_mecha
%pip install -q -e .
%cd /content

/content/sd_mecha
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 71.9 MB/s eta 0:00:00
  Building editable for sd-mecha (pyproject.toml) ... done
/content


In [ ]:
from pathlib import Path

model_a_path = "/content/drive/MyDrive/Comfy/ComfyUI/models/checkpoints/SDXL/Anti_Pony_Test.safetensors" #@param{type:'string'}
model_b_path = "/content/drive/MyDrive/Comfy/ComfyUI/models/checkpoints/SDXL/illustriousXL10_v10.safetensors" #@param{type:'string'}
output_path = "rebase_1.safetensors" #@param{type:'string'}
output_dir = "/content/drive/MyDrive/sd_mecha_outputs" #@param{type:'string'}

if not osp.exists(output_dir):
    os.makedirs(output_dir)

model_a = Path(model_a_path)
model_b = Path(model_b_path)

In [ ]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()
import sd_mecha

a = sd_mecha.model(model_a)
b = sd_mecha.model(model_b)

sgm_a = sd_mecha.convert(a, "sdxl-sgm_split")
sgm_b = sd_mecha.convert(b, "sdxl-sgm_split")

#rebasin_cache = {}

rebasin = sd_mecha.sdxl_sgm_split_rebasin(
    sgm_a,
    sgm_b,
    iters=10,
).set_cache()

rebasin_sdxl = sd_mecha.convert(rebasin, "sdxl-sgm")


sdxl_config = sd_mecha.extensions.model_configs.resolve("sdxl-sgm")

def get_clip_last_block_alpha(k):
    return 1.0 if k.startswith((
        "conditioner.embedders.1.model.transformer.resblocks.31",
        "conditioner.embedders.0.transformer.text_model.encoder.layers.11",
    )) else 0.0

recipe = sd_mecha.weighted_sum(rebasin_sdxl, b, {k: get_clip_last_block_alpha(k) for k in sdxl_config.keys()})

output_full = str(Path(output_dir) / output_path)

sd_mecha.merge(
    recipe,
    output=output_full,
    merge_device="cuda",
    output_device="cpu",
    check_finite=True,
    total_buffer_size=2**26,
    threads=0,
)


In [ ]:
# works!
import sd_mecha
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
#device = torch.device("cuda")

a = sd_mecha.model(model_a_path)
b = sd_mecha.model(model_b_path)

rebasin_recipe = sd_mecha.convert(
    sd_mecha.sdxl_sgm_split_rebasin(
        sd_mecha.convert(a, "sdxl-sgm_split"),
        sd_mecha.convert(b, "sdxl-sgm_split"),
    ).set_cache(),
    "sdxl-sgm",
)

sdxl_config = sd_mecha.extensions.model_configs.resolve("sdxl-sgm")

def get_clip_last_block_alpha(k):
    return 1.0 if k.startswith((
        "conditioner.embedders.1.model.transformer.resblocks.31",
        "conditioner.embedders.0.transformer.text_model.encoder.layers.11",
        "model.diffusion_model.input_blocks.8.1.transformer_blocks.3.attn2.to_q.weight",

    )) else 0.0

recipe = sd_mecha.weighted_sum(rebasin_recipe, b, {k: get_clip_last_block_alpha(k) for k in sdxl_config.keys()})

output_full = str(Path(output_dir) / output_path)

sd_mecha.merge(
    recipe,
    output=output_full,
    merge_dtype=torch.float32,
    merge_device="cuda",
    output_device="cpu",
    check_finite=True,
    total_buffer_size=2**26,
    threads=0,
)


Merging recipe: 100%|██████████| 2519/2519 [06:16<00:00,  6.69it/s, key=ztsnr, shape=[0]]
